# Generalized DBSCAN clustering

Clusters all feature names together, without filtering by product type.

Like the other clustering notebooks, this one loads a precomputed similarity matrix, set by `MATRIX_PATH` in the setup cell (`data/similarity.npz` from `3_similarity.ipynb`, or `data/similarity_expanded.npz` from `3b_similarity_expanded.ipynb`). This keeps the input fixed, so result differences come from the algorithm itself.

**Method**:
1. Convert similarity to distance (`distance = 1 - similarity`).
2. Try a compact grid of DBSCAN parameters (`eps`, `min_samples`).
3. Select the configuration with the best silhouette score (noise points counted as singleton clusters) among those with at least `MIN_COVERAGE` of the items clustered.
4. Inspect clusters and visualize with t-SNE.
5. Check parameter sensitivity (k-distance plot, grid heatmaps).

## Definitions used in this notebook

Notation follows the Week 6 slides. A **feature** is one property on a product page (feature name + description); we cluster the **feature names**. $S$ is the set of the $n$ feature names, and `labels` below are cluster labels (`-1` = noise).

**Distance and $\varepsilon$.** The similarity matrix $\mathrm{sim}(p,x)\in[0,1]$ is turned into a distance

$$d(p,x) := 1 - \mathrm{sim}(p,x).$$

$\varepsilon$ is a **distance threshold**: two feature names are neighbours if $d(p,x)\le\varepsilon$, i.e. $\mathrm{sim}(p,x)\ge 1-\varepsilon$. A larger $\varepsilon$ is a looser rule. $m$ is `min_samples`.

**Reachability criteria** (slides 52 to 55)

- $p\in S$ is a **core point** iff $\bigl|\{x\in S \mid d(p,x)\le\varepsilon\}\bigr|\ge m$ (the point itself counts).
- $x$ is **directly reachable** from $p$ iff $d(p,x)\le\varepsilon$ and $p$ is a core point.
- $x$ is **reachable** from $p$ iff there is a chain $x_1=p,\dots,x_k=x$ in which every $x_{i+1}$ is directly reachable from $x_i$ (so $x_1,\dots,x_{k-1}$ are core points).
- A reachable point that is not core is a **rim point**; a point not reachable from any other point is **noise** (label `-1`).
- A **cluster** is built from a core point $p$ as $C := \{x \mid x \text{ reachable from } p \text{ and not already in a cluster}\}$.

**Score heuristic** (selection of $\varepsilon$ and $m$). Every noise point is relabelled as its own one-point cluster with silhouette $0$. For a clustered point $p\in C_i$,

$$a(p)=\frac{1}{|C_i|-1}\sum_{x\in C_i,\,x\ne p} d(p,x),\qquad b(p)=\min_{j\ne i}\frac{1}{|C_j|}\sum_{x\in C_j} d(p,x),\qquad s(p)=\frac{b(p)-a(p)}{\max\{a(p),\,b(p)\}},$$

$$\text{score}=\frac1n\sum_{p\in S} s(p).$$

Candidates with fewer than 2 clusters, or with coverage $=|\{p:\text{label}(p)\ne-1\}|/n$ below `MIN_COVERAGE`, are dropped; the rest are ranked by score (ties: higher coverage, then fewer clusters) and the best is selected. The silhouette over the clustered points only (`silhouette_non_noise`) and the Davies-Bouldin index are reported for reference and are not used for selection. The silhouette is due to Rousseeuw (1987).

> ## ⚠️ Additional constraint: minimum coverage (`MIN_COVERAGE`)
>
> **Standard DBSCAN has no coverage requirement. We add one.** A candidate $(\varepsilon, m)$ is only eligible if
>
> $$\text{coverage}=\frac{|\{p\in S:\ \text{label}(p)\ne -1\}|}{n}\;\ge\;\texttt{MIN\_COVERAGE}\quad(=0.50).$$
>
> **Why.** Because noise points count as silhouette $0$, the score factors as $\text{score}=\text{coverage}\times\bar S_{g}'$, where $\bar S_g'$ is the mean silhouette of the clustered points. A small $\varepsilon$ can therefore win by clustering only a few tight pairs (high $\bar S_g'$, low coverage) and discarding most feature names as noise. The floor removes those candidates, so the selected clustering both scores well and groups at least half of the feature names.
>
> **Effect on the selection.** The rule is applied before ranking: candidates below the floor are labelled `dropped: coverage below 50%` in the search cell, and the remaining ones are ranked by score. On the expanded matrix (664 feature names) the best unconstrained candidate ($\varepsilon=0.40$, score 0.126) groups only 37% of the feature names; with the floor the selected setting is $\varepsilon=0.48$, $m=2$ (score 0.078, coverage 51.8%). The floor trades a lower raw score for a clustering that actually covers the catalogue. Raise or lower it with `MIN_COVERAGE` in the setup cell.


In [1]:
from collections import Counter

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display
from sklearn.cluster import DBSCAN
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

EPS_VALUES = np.round(np.arange(0.20, 0.82, 0.02), 2)  # original grid (colleagues)
MIN_SAMPLES_VALUES = [2, 3, 4, 5, 6, 8]
MIN_COVERAGE = 0.50  # relaxed floor: minimum share of feature names that must not be noise
SEED = 0
MATRIX_PATH = "data/similarity_qwen3_8b_api.npz"  # use "data/similarity.npz" for the original 445 feature names
SHOW_ALL_RESULTS = False  # True prints every eps x min_samples combination
HIGHLIGHT_GROUPS = 20

with np.load(MATRIX_PATH) as data:
    similarity = data["similarity"]
    feature_names = data["titles"].tolist()

n = len(feature_names)
assert similarity.shape == (n, n)
assert np.allclose(similarity, similarity.T), "similarity must be symmetric"

distance_matrix = 1 - similarity
distance_matrix = np.clip(distance_matrix, 0, 1)
np.fill_diagonal(distance_matrix, 0)

print(f"Loaded {n} feature names, matrix {similarity.shape}")

Loaded 445 feature names, matrix (445, 445)


## 1. Parameter search

For each `(eps, min_samples)` pair, we fit DBSCAN on the precomputed distance matrix.

Parameters are selected by the silhouette score computed on all feature names, with each noise point treated as its own singleton cluster (silhouette 0). This prevents a configuration from scoring well by discarding difficult feature names as noise. Ties are broken by higher coverage, then fewer clusters. Both methods search the same `min_samples` range; DBSCAN additionally searches `eps` and HDBSCAN `min_cluster_size`. HDBSCAN has no `eps` parameter.

The silhouette on non-noise points only is reported for reference but is not used for selection.

In [2]:
def noise_as_singletons(labels):
    relabeled = labels.copy()
    noise_indices = np.where(relabeled == -1)[0]
    start = relabeled.max() + 1
    relabeled[noise_indices] = np.arange(start, start + len(noise_indices))
    return relabeled


def evaluate(labels, D):
    is_noise = labels == -1
    n_noise = int(is_noise.sum())
    n_clusters = len(set(labels[~is_noise].tolist()))
    if n_clusters < 2:
        return None

    singleton_labels = noise_as_singletons(labels)
    if len(set(singleton_labels.tolist())) >= len(labels):
        return None
    score = silhouette_score(D, singleton_labels, metric="precomputed")

    non_noise_labels = labels[~is_noise]
    if n_clusters < len(non_noise_labels):
        silhouette_non_noise = silhouette_score(
            D[np.ix_(~is_noise, ~is_noise)], non_noise_labels, metric="precomputed"
        )
    else:
        silhouette_non_noise = float("nan")

    return {
        "score": float(score),
        "silhouette_non_noise": float(silhouette_non_noise),
        "coverage": float((~is_noise).mean()),
        "n_clusters": int(n_clusters),
        "n_noise": n_noise,
    }


def format_row(row, param_names):
    params = ", ".join(f"{name}={row[name]:g}" for name in param_names)
    return (
        f"{params} | "
        f"score={row['score']:.4f}, silhouette_non_noise={row['silhouette_non_noise']:.4f}, "
        f"clusters={row['n_clusters']}, coverage={100 * row['coverage']:.1f}%, noise={row['n_noise']}"
    )


def warn_if_at_grid_edge(best, grids):
    for name, grid in grids.items():
        at_lower_edge = np.isclose(best[name], min(grid)) and name != "min_samples"  # min_samples cannot go below 2
        if at_lower_edge or np.isclose(best[name], max(grid)):
            print(f"Warning ({name}={best[name]:g}): Selected value is at the edge of the grid - consider extending the grid.")


PARAM_GRIDS = {"eps": EPS_VALUES, "min_samples": MIN_SAMPLES_VALUES}

TOP_N = 10

all_results = []  # every eps x min_samples combination, with the reason it was kept or dropped
candidates = []
for eps in EPS_VALUES:
    for min_samples in MIN_SAMPLES_VALUES:
        model = DBSCAN(eps=float(eps), min_samples=min_samples, metric="precomputed")
        labels = model.fit_predict(distance_matrix)
        evaluation = evaluate(labels, distance_matrix)
        row = {"eps": float(eps), "min_samples": int(min_samples)}
        if evaluation is None:
            noise_mask = labels == -1
            row.update(
                score=float("nan"), silhouette_non_noise=float("nan"),
                coverage=float((~noise_mask).mean()),
                n_clusters=len(set(labels[~noise_mask].tolist())), n_noise=int(noise_mask.sum()),
                status="dropped: fewer than 2 clusters",
            )
        else:
            row.update(evaluation, status="ok")
            candidates.append(row)
        all_results.append(row)

if not candidates:
    raise ValueError("No valid DBSCAN configuration produced at least 2 non-noise clusters.")

# Selection rule:
#   1. drop configurations with fewer than 2 non-noise clusters (silhouette undefined)
#   2. drop configurations whose coverage is below MIN_COVERAGE
#   3. rank the rest by score (silhouette, noise as singletons), highest first
#   4. ties: higher coverage first, then fewer clusters
#   The best-ranked configuration is selected; the TOP_N best-ranked are printed.
for row in all_results:
    if row["status"] == "ok" and row["coverage"] < MIN_COVERAGE:
        row["status"] = f"dropped: coverage below {MIN_COVERAGE:.0%}"

eligible = [r for r in candidates if r["coverage"] >= MIN_COVERAGE]
if not eligible:
    raise ValueError(f"No configuration reaches coverage >= {MIN_COVERAGE:.0%}; lower MIN_COVERAGE or raise eps.")
ranked = sorted(eligible, key=lambda r: (-r["score"], -r["coverage"], r["n_clusters"]))
best = ranked[0]
candidates = eligible  # the heatmaps below only show configurations that passed the filters

status_counts = Counter(row["status"] for row in all_results)
print(f"{len(all_results)} combinations tried ({len(EPS_VALUES)} eps x {len(MIN_SAMPLES_VALUES)} min_samples):")
for status, count in status_counts.most_common():
    print(f"  {count:4d} {status}")
if SHOW_ALL_RESULTS:
    for row in sorted(all_results, key=lambda r: (r["eps"], r["min_samples"])):
        print(
            f"  eps={row['eps']:g}, min_samples={row['min_samples']} | "
            f"clusters={row['n_clusters']}, coverage={100 * row['coverage']:.1f}%, noise={row['n_noise']} | "
            f"score={row['score']:.4f} | {row['status']}"
        )

print(f"\n{len(eligible)} of {len(all_results)} passed the filters. Top {min(TOP_N, len(ranked))} by score:")
for row in ranked[:TOP_N]:
    print(format_row(row, PARAM_GRIDS))

print("\nSelected configuration:")
print(format_row(best, PARAM_GRIDS))
warn_if_at_grid_edge(best, PARAM_GRIDS)

186 combinations tried (31 eps x 6 min_samples):
    68 dropped: coverage below 50%
    65 ok
    53 dropped: fewer than 2 clusters

65 of 186 passed the filters. Top 10 by score:
eps=0.46, min_samples=2 | score=0.1365, silhouette_non_noise=0.3589, clusters=64, coverage=53.0%, noise=209
eps=0.48, min_samples=2 | score=0.1212, silhouette_non_noise=0.3179, clusters=60, coverage=57.1%, noise=191
eps=0.5, min_samples=2 | score=0.0988, silhouette_non_noise=0.2478, clusters=61, coverage=63.6%, noise=162
eps=0.52, min_samples=2 | score=0.0686, silhouette_non_noise=0.2102, clusters=53, coverage=66.1%, noise=151
eps=0.54, min_samples=2 | score=0.0318, silhouette_non_noise=0.1379, clusters=47, coverage=70.8%, noise=130
eps=0.78, min_samples=2 | score=0.0173, silhouette_non_noise=0.0435, clusters=2, coverage=98.9%, noise=5
eps=0.78, min_samples=3 | score=0.0173, silhouette_non_noise=0.0435, clusters=2, coverage=98.9%, noise=5
eps=0.8, min_samples=2 | score=0.0173, silhouette_non_noise=0.0435, clu

## 2. Fit DBSCAN and inspect groups

In [3]:
final_model = DBSCAN(eps=best["eps"], min_samples=best["min_samples"], metric="precomputed")
labels = final_model.fit_predict(distance_matrix)

is_noise = labels == -1
cluster_labels = [label for label in sorted(set(labels.tolist())) if label != -1]
sizes = Counter(labels)

print(
    f"Clusters (excluding noise): {len(cluster_labels)} | "
    f"noise points: {sizes.get(-1, 0)} / {len(labels)} ({100 * sizes.get(-1, 0) / len(labels):.1f}%)"
)
print(f"Cluster sizes (largest first): {[size for label, size in sizes.most_common() if label != -1]}")

members_by_cluster = {
    label: [feature_names[index] for index, assigned_label in enumerate(labels) if assigned_label == label]
    for label in cluster_labels
}

# for label, members in sorted(members_by_cluster.items(), key=lambda item: (-len(item[1]), item[0])):
#     if len(members) > 1:
#         print(f"\nCluster {label} ({len(members)} feature names)")
#         for member in members:
#             print(f"  - {member}")

Clusters (excluding noise): 64 | noise points: 209 / 445 (47.0%)
Cluster sizes (largest first): [15, 12, 11, 10, 9, 9, 8, 6, 6, 6, 6, 5, 5, 5, 5, 5, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]


## 3. Cluster visualization

t-SNE projects precomputed distances (`1 - similarity`) to two dimensions for display only.
The largest groups receive highlight colors, smaller groups are grey, and noise points are hollow markers.

In [4]:
INK = "#2b2b2b"
GRAY = "#d7d7d7"
LIGHT_GRAY = "#c9c8c0"
HIGHLIGHT_PALETTE = [  # hard-coded on purpose: one distinct colour per highlighted group, so growing eps never runs out
    "#cc1f1a", "#005f73", "#0a9396", "#ee9b00", "#ae2012", "#6c757d", "#9b2226", "#3a86ff",
    "#7209b7", "#2d6a4f", "#f72585", "#8d99ae", "#ff7f11", "#4361ee", "#6a994e", "#b5179e",
    "#264653", "#e9c46a", "#9c6644", "#118ab2",
]
assert len(HIGHLIGHT_PALETTE) >= HIGHLIGHT_GROUPS, "add colours or lower HIGHLIGHT_GROUPS"

perplexity = min(30, len(feature_names) - 1)
coordinates = TSNE(
    n_components=2,
    metric="precomputed",
    init="random",
    random_state=SEED,
    perplexity=perplexity,
).fit_transform(distance_matrix)

multi = [label for label in cluster_labels if sizes[label] > 1]
ranked_groups = sorted(multi, key=lambda label: -sizes[label])
highlighted_groups = ranked_groups[:HIGHLIGHT_GROUPS]

def group_name(label):
    indices = np.where(labels == label)[0]
    within_group_similarity = similarity[np.ix_(indices, indices)]
    medoid_index = indices[within_group_similarity.sum(axis=1).argmax()]
    return feature_names[medoid_index]

group_names = {label: group_name(label) for label in multi}

def make_trace(indices, name, color, custom_group_names, hollow=False, size=9):
    if len(indices) == 0:
        return None
    return go.Scatter(
        x=coordinates[indices, 0],
        y=coordinates[indices, 1],
        mode="markers",
        name=name,
        marker=dict(
            size=size,
            color="white" if hollow else color,
            line=dict(width=1.5 if hollow else 1, color=color if hollow else "white"),
        ),
        text=[feature_names[index] for index in indices],
        customdata=[custom_group_names[index] for index in indices],
        hovertemplate="<b>%{text}</b><br>group: %{customdata}<extra></extra>",
    )

custom_group_names = []
for label in labels:
    if label == -1:
        custom_group_names.append("Noise")
    else:
        custom_group_names.append(group_names.get(label, f"Cluster {label}"))

noise_indices = np.where(labels == -1)[0]
other_group_indices = np.array(
    [index for index, label in enumerate(labels) if label != -1 and sizes[label] > 1 and label not in highlighted_groups],
    dtype=int,
)

figure = go.Figure()

trace = make_trace(noise_indices, f"Noise ({len(noise_indices)})", LIGHT_GRAY, custom_group_names, hollow=True, size=7)
if trace is not None:
    figure.add_trace(trace)

trace = make_trace(
    other_group_indices,
    f"Other groups ({max(len(ranked_groups) - len(highlighted_groups), 0)})",
    GRAY,
    custom_group_names,
    size=7,
)
if trace is not None:
    figure.add_trace(trace)

for color, label in zip(HIGHLIGHT_PALETTE, highlighted_groups):
    indices = np.where(labels == label)[0]
    trace = make_trace(indices, f"{group_names[label]} ({sizes[label]})", color, custom_group_names, size=10)
    if trace is not None:
        figure.add_trace(trace)

figure.update_layout(
    title=dict(text="Feature-name clusters: DBSCAN (t-SNE)", x=0, font=dict(color=INK)),
    template="simple_white",
    width=1000,
    height=720,
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    legend=dict(title="Largest groups", font=dict(color=INK), itemsizing="constant"),
    hoverlabel=dict(bgcolor="white", font_color=INK),
)
try:
    figure.show(renderer="notebook_connected")
except ValueError:
    display(HTML(figure.to_html(full_html=False, include_plotlyjs=True)))

### 3.1 All tested eps values

One t-SNE panel for every 5th `eps` in the grid (plus the selected one; `SWEEP_PANEL_STEP` changes this), at the selected `min_samples`. The t-SNE coordinates are reused from the cell above, so only the cluster labels change between panels. Each panel title gives the number of clusters, the largest cluster, how many items are grouped (in a cluster, with %) and how many are noise (singletons, with %). Hollow grey points are noise; every cluster has its own colour (colours repeat past the last entry of `HIGHLIGHT_PALETTE`).


In [5]:
sweep_min_samples = best["min_samples"]
SWEEP_PANEL_STEP = 5  # draw every 5th eps of the grid (the selected eps is always drawn); the table below lists all
n_cols = 3

sweep_labels = {
    float(eps): DBSCAN(eps=float(eps), min_samples=sweep_min_samples, metric="precomputed").fit_predict(distance_matrix)
    for eps in EPS_VALUES
}
panel_eps = sorted(set(list(sweep_labels)[::SWEEP_PANEL_STEP]) | {float(best["eps"])})
panel_labels = {eps: sweep_labels[eps] for eps in panel_eps}
n_rows = int(np.ceil(len(panel_labels) / n_cols))

def sweep_title(eps, labels_eps):
    noise = int((labels_eps == -1).sum())
    n_clusters = len(set(labels_eps.tolist()) - {-1})
    largest = max(Counter(labels_eps[labels_eps != -1].tolist()).values(), default=0)
    grouped = len(labels_eps) - noise
    return (
        f"eps={eps:g}: {n_clusters} clusters, largest {largest}"
        f"<br>grouped {grouped} ({100 * grouped / len(labels_eps):.1f}%) | noise {noise} ({100 * noise / len(labels_eps):.1f}%)"
    )

sweep_figure = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=[sweep_title(eps, lab) for eps, lab in panel_labels.items()],
    horizontal_spacing=0.02, vertical_spacing=0.12,
)

palette = HIGHLIGHT_PALETTE  # same hard-coded colours as the t-SNE plot; repeats past the last colour
for panel, (eps, labels_eps) in enumerate(panel_labels.items()):
    row, col = panel // n_cols + 1, panel % n_cols + 1
    noise_idx = np.where(labels_eps == -1)[0]
    sweep_figure.add_trace(
        go.Scatter(
            x=coordinates[noise_idx, 0], y=coordinates[noise_idx, 1], mode="markers",
            marker=dict(size=5, color="white", line=dict(width=1, color=LIGHT_GRAY)),
            text=[feature_names[i] for i in noise_idx], hovertemplate="%{text}<extra>noise</extra>",
            showlegend=False,
        ),
        row=row, col=col,
    )
    for k, cluster in enumerate(sorted(set(labels_eps.tolist()) - {-1})):
        idx = np.where(labels_eps == cluster)[0]
        sweep_figure.add_trace(
            go.Scatter(
                x=coordinates[idx, 0], y=coordinates[idx, 1], mode="markers",
                marker=dict(size=6, color=palette[k % len(palette)], line=dict(width=0.5, color="white")),
                text=[feature_names[i] for i in idx],
                hovertemplate="%{text}<extra>cluster " + str(cluster) + "</extra>",
                showlegend=False,
            ),
            row=row, col=col,
        )

sweep_figure.update_xaxes(visible=False)
sweep_figure.update_yaxes(visible=False)
sweep_figure.update_layout(
    title=dict(text=f"DBSCAN across eps (min_samples={sweep_min_samples}, t-SNE)", x=0),
    template="simple_white", width=1200, height=420 * n_rows,
)
try:
    sweep_figure.show(renderer="notebook_connected")
except ValueError:
    display(HTML(sweep_figure.to_html(full_html=False, include_plotlyjs=True)))


#### Summary table for the tested eps values

The same runs as the panels above, one row per `eps`, all at the selected `min_samples` (shown in its own column). `Score` is the selection score (silhouette with noise as singletons); `Silhouette (grouped only)` scores only the items that are in a cluster. Both are blank when there are fewer than 2 clusters. The selected `eps` is highlighted.

In [6]:
sweep_rows = []
for eps, labels_eps in sweep_labels.items():
    noise = int((labels_eps == -1).sum())
    cluster_sizes = Counter(labels_eps[labels_eps != -1].tolist())
    metrics = evaluate(labels_eps, distance_matrix)
    sweep_rows.append({
        "eps": eps,
        "min_samples": sweep_min_samples,
        "clusters": len(cluster_sizes),
        "largest": max(cluster_sizes.values(), default=0),
        "grouped": len(labels_eps) - noise,
        "grouped_pct": 100 * (len(labels_eps) - noise) / len(labels_eps),
        "noise": noise,
        "noise_pct": 100 * noise / len(labels_eps),
        "score": metrics["score"] if metrics else float("nan"),
        "silhouette_non_noise": metrics["silhouette_non_noise"] if metrics else float("nan"),
    })


def fmt(value, spec):
    return "" if value != value else format(value, spec)  # NaN -> blank


columns = [
    ("eps", "eps", ".2f"),
    ("min_samples", "min_samples", "d"),
    ("Clusters", "clusters", "d"),
    ("Largest cluster", "largest", "d"),
    ("Grouped", "grouped", "d"),
    ("Grouped %", "grouped_pct", ".1f"),
    ("Noise", "noise", "d"),
    ("Noise %", "noise_pct", ".1f"),
    ("Score", "score", ".4f"),
    ("Silhouette (grouped only)", "silhouette_non_noise", ".4f"),
]
selected = [bool(np.isclose(row["eps"], best["eps"])) for row in sweep_rows]
fill = ["#fde8e7" if flag else "white" for flag in selected]

table_figure = go.Figure(
    go.Table(
        header=dict(values=[title for title, _, _ in columns], fill_color="#2b2b2b", font=dict(color="white"), align="center"),
        cells=dict(
            values=[[fmt(row[key], spec) for row in sweep_rows] for _, key, spec in columns],
            fill_color=[fill] * len(columns),
            align="center",
            height=28,
        ),
    )
)
table_figure.update_layout(
    title=dict(text=f"DBSCAN across eps (min_samples={sweep_min_samples}); selected eps={best['eps']:g} highlighted", x=0),
    width=1000, height=140 + 30 * len(sweep_rows), margin=dict(t=60, b=10),
)
try:
    table_figure.show(renderer="notebook_connected")
except ValueError:
    display(HTML(table_figure.to_html(full_html=False, include_plotlyjs=True)))


## 5. Evaluation

### 5.1 Internal quality

DBSCAN marks some points as noise (`-1`). The selection score treats each noise point as its own singleton cluster (silhouette 0), so discarding feature names as noise does not inflate it. The silhouette on non-noise points only is shown for reference.

In [7]:
metrics = evaluate(labels, distance_matrix)

if metrics is None:
    print("Silhouette score not defined (need at least 2 non-noise clusters).")
else:
    print(f"Silhouette score (noise as singletons, selection score): {metrics['score']:.4f}")
    print(f"Silhouette score (non-noise only, reported only): {metrics['silhouette_non_noise']:.4f}")
    print(f"Clusters (excluding noise): {metrics['n_clusters']}")
    print(f"Coverage: {100 * metrics['coverage']:.1f}%")
    print(f"Noise points: {metrics['n_noise']} / {len(labels)}")

Silhouette score (noise as singletons, selection score): 0.1365
Silhouette score (non-noise only, reported only): 0.3589
Clusters (excluding noise): 64
Coverage: 53.0%
Noise points: 209 / 445


#### Davies-Bouldin index (course metric)

The lecture evaluates clusterings with the Davies-Bouldin index. The data has no coordinates, so each cluster is represented by its **clustroid**, the member with the smallest average distance to the other members, instead of a centroid. With clustroid $c_i$, average radius $r_i = \frac{1}{|C_i|}\sum_{c\in C_i} d(c_i, c)$:

$$DB = \frac1k \sum_{i=1}^{k} \max_{j\ne i} \frac{r_i + r_j}{d(c_i, c_j)}$$

Lower is better. Noise points are left out (they belong to no cluster), so read it together with coverage.

In [8]:
def davies_bouldin(labels, D):
    cluster_ids = sorted(set(labels.tolist()) - {-1})
    if len(cluster_ids) < 2:
        return None
    clustroids, radii = [], []
    for cluster_id in cluster_ids:
        members = np.where(labels == cluster_id)[0]
        within = D[np.ix_(members, members)]
        clustroid = members[within.mean(axis=1).argmin()]  # smallest average distance to the other members
        clustroids.append(clustroid)
        radii.append(D[clustroid, members].mean())
    radii = np.array(radii)
    total = 0.0
    for i in range(len(cluster_ids)):
        others = [j for j in range(len(cluster_ids)) if j != i]
        total += max((radii[i] + radii[j]) / D[clustroids[i], clustroids[j]] for j in others)
    return total / len(cluster_ids)


db_index = davies_bouldin(labels, distance_matrix)
if db_index is None:
    print("Davies-Bouldin index not defined (need at least 2 non-noise clusters).")
else:
    print(f"Davies-Bouldin index (clustroids, noise excluded, lower is better): {db_index:.4f}")


Davies-Bouldin index (clustroids, noise excluded, lower is better): 0.8764


### 5.2 Parameter selection

These plots show how sensitive the result is to the chosen parameters, using the results already computed in the parameter search (no refitting).

- **k-distance plot** has one line per `min_samples` value $m$ (the selected one is thick). Each line sorts every feature name's distance to its $k=m-1$ nearest other feature name. A feature name can be a core point at $\varepsilon$ when its line lies at or below $\varepsilon$, so the horizontal lines (dotted for the other $\varepsilon$ values of the sweep, red dashed for the selected one) show how many can be core. The table printed under the plot gives these shares for every $m$ and $\varepsilon$. A line that bends sharply upward marks where feature names stop having close neighbours.
- **Heatmaps** show the selection score, coverage and number of clusters for every configuration in the grid. The colour scale is Viridis, from dark blue-purple (low) to yellow (high), and each panel has its own scale (see its colour bar). The **red square** is not part of the scale: it only marks the selected configuration. A selected cell inside a broad yellow area means the choice is stable; an isolated peak, or one at the grid edge, means it depends on the exact values. Blank cells were dropped: fewer than 2 clusters, or coverage below `MIN_COVERAGE`.
- **Sensitivity summary** lists the score and number of clusters one grid step away from the selected configuration in each direction. Small changes in score but large changes in cluster count mean the score alone does not pin down the granularity.

_Placeholder: DBSCAN finds N clusters with the selected parameters, compared with k = … chosen for spectral clustering in `5_spectral.ipynb`._

In [9]:
sorted_distances = np.sort(distance_matrix, axis=1)  # column 0 is the point itself (distance 0)
selected_m = best["min_samples"]

figure = go.Figure()
for line_index, m in enumerate(MIN_SAMPLES_VALUES):
    k = m - 1  # a core point needs m points within eps, itself included, i.e. k = m - 1 other points
    k_distances = np.sort(sorted_distances[:, k])
    figure.add_trace(
        go.Scatter(
            x=np.arange(1, len(k_distances) + 1), y=k_distances, mode="lines",
            line=dict(color=HIGHLIGHT_PALETTE[line_index], width=3.5 if m == selected_m else 1.5),
            name=f"m = {m} (k = {k})" + (" selected" if m == selected_m else ""),
            hovertemplate="rank %{x}<br>distance %{y:.3f}<extra>m = " + str(m) + "</extra>",
        )
    )
for eps in panel_eps:  # every eps drawn in the t-SNE panels, plus the selected one
    is_selected = bool(np.isclose(eps, best["eps"]))
    figure.add_hline(
        y=eps, opacity=1,
        line=dict(color="#cc1f1a" if is_selected else "#c9c8c0", dash="dash" if is_selected else "dot", width=2 if is_selected else 1),
        annotation_text=f"eps = {eps:g}" + (" (selected)" if is_selected else ""),
        annotation_position="top left", annotation_font_size=10,
    )
figure.update_layout(
    title=dict(text="k-distance plot for every min_samples (k = min_samples - 1)", x=0),
    template="simple_white", width=1000, height=520,
    xaxis_title="Feature names, sorted by distance", yaxis_title="Distance to k-th nearest other feature name",
    legend=dict(title="min_samples"),
)
try:
    figure.show(renderer="notebook_connected")
except ValueError:
    display(HTML(figure.to_html(full_html=False, include_plotlyjs=True)))

print("Share of feature names that can be core points (k-distance <= eps):")
print(f"{'m':>3} {'k':>2} | " + " ".join(f"eps={eps:<5g}" for eps in panel_eps))
for m in MIN_SAMPLES_VALUES:
    k_distances = sorted_distances[:, m - 1]
    shares = " ".join(f"{(k_distances <= eps).mean():>9.1%}" for eps in panel_eps)
    print(f"{m:>3} {m - 1:>2} | {shares}")


Share of feature names that can be core points (k-distance <= eps):
  m  k | eps=0.2   eps=0.3   eps=0.4   eps=0.46  eps=0.5   eps=0.6   eps=0.7   eps=0.8  
  2  1 |     10.3%     20.4%     39.3%     53.0%     63.6%     85.2%     95.7%     98.9%
  3  2 |      1.8%      4.9%     17.5%     27.2%     38.2%     70.8%     92.1%     98.2%
  4  3 |      0.0%      1.1%      7.0%     16.0%     23.4%     56.9%     87.4%     96.4%
  5  4 |      0.0%      0.0%      1.3%      8.5%     17.1%     47.6%     82.2%     95.5%
  6  5 |      0.0%      0.0%      0.0%      2.7%     11.2%     37.3%     77.1%     93.3%
  8  7 |      0.0%      0.0%      0.0%      0.4%      2.7%     23.6%     64.9%     89.2%


In [10]:
SELECTED_COLOR = "#cc1f1a"
HEATMAP_COLORSCALE = "Viridis"
HEATMAP_METRICS = {
    "score": ("Score (silhouette, noise as singletons)", ".3f"),
    "coverage": ("Coverage (%)", ".1f"),
    "n_clusters": ("Clusters", ".0f"),
}


def grid_table(candidates, x_name, x_values, y_name, y_values):
    lookup = {(row[x_name], row[y_name]): row for row in candidates}
    table = {metric: np.full((len(y_values), len(x_values)), np.nan) for metric in HEATMAP_METRICS}
    for j, x in enumerate(x_values):
        for i, y in enumerate(y_values):
            row = lookup.get((x, y))
            if row is None:
                continue
            for metric in HEATMAP_METRICS:
                table[metric][i, j] = 100 * row[metric] if metric == "coverage" else row[metric]
    return table


def plot_param_heatmaps(table, x_name, x_values, y_name, y_values, best, method):
    x_labels = [f"{x:g}" for x in x_values]
    y_labels = [f"{y:g}" for y in y_values]
    figure = make_subplots(
        rows=len(HEATMAP_METRICS), cols=1,
        subplot_titles=[title for title, _ in HEATMAP_METRICS.values()],
        vertical_spacing=0.1,
    )
    for row_index, (metric, (title, fmt)) in enumerate(HEATMAP_METRICS.items(), start=1):
        domain = figure.get_subplot(row_index, 1).yaxis.domain
        figure.add_trace(
            go.Heatmap(
                z=table[metric], x=x_labels, y=y_labels,
                colorscale=HEATMAP_COLORSCALE, hoverongaps=False,
                colorbar=dict(len=domain[1] - domain[0], y=(domain[0] + domain[1]) / 2, thickness=12),
                hovertemplate=f"{x_name}=%{{x}}<br>{y_name}=%{{y}}<br>{title}: %{{z:{fmt}}}<extra></extra>",
            ),
            row=row_index, col=1,
        )
        figure.add_trace(
            go.Scatter(
                x=[f"{best[x_name]:g}"], y=[f"{best[y_name]:g}"], mode="markers",
                marker=dict(symbol="square-open", size=18, color=SELECTED_COLOR, line=dict(width=3)),
                name="Selected", showlegend=row_index == 1, hoverinfo="skip",
            ),
            row=row_index, col=1,
        )
        figure.update_xaxes(
            title_text=x_name if row_index == len(HEATMAP_METRICS) else None,
            type="category", tickangle=-45 if len(x_values) > 12 else 0, row=row_index, col=1,
        )
        figure.update_yaxes(title_text=y_name, type="category", row=row_index, col=1)
    figure.update_layout(
        title=dict(text=f"{method} parameter grid (blank = dropped: fewer than 2 clusters or coverage below {MIN_COVERAGE:.0%})", x=0),
        template="simple_white", width=1000, height=1000,
        legend=dict(orientation="h", x=1, xanchor="right", y=1.04),
    )
    show(figure)


def print_sensitivity(candidates, best, grids):
    names = list(grids)
    lookup = {tuple(row[name] for name in names): row for row in candidates}
    print(f"Selected: {format_row(best, grids)}")
    print("\nOne grid step away (score change vs selected):")
    for name, grid in grids.items():
        grid = list(grid)
        index = int(np.argmin([abs(value - best[name]) for value in grid]))
        for step in (-1, 1):
            if not 0 <= index + step < len(grid):
                print(f"  {name}: no {'lower' if step < 0 else 'higher'} value in grid")
                continue
            key = tuple(grid[index + step] if other == name else best[other] for other in names)
            params = ", ".join(f"{other}={value:g}" for other, value in zip(names, key))
            row = lookup.get(key)
            if row is None:
                print(f"  {params} | fewer than 2 clusters or not evaluated")
            else:
                print(
                    f"  {params} | score={row['score']:.4f} ({row['score'] - best['score']:+.4f}), "
                    f"clusters={row['n_clusters']}"
                )


def show(figure):
    try:
        figure.show(renderer="notebook_connected")
    except ValueError:
        display(HTML(figure.to_html(full_html=False, include_plotlyjs=True)))


grids = {"eps": EPS_VALUES.tolist(), "min_samples": MIN_SAMPLES_VALUES}
table = grid_table(candidates, "eps", grids["eps"], "min_samples", grids["min_samples"])
plot_param_heatmaps(table, "eps", grids["eps"], "min_samples", grids["min_samples"], best, "DBSCAN")
print_sensitivity(candidates, best, grids)

Selected: eps=0.46, min_samples=2 | score=0.1365, silhouette_non_noise=0.3589, clusters=64, coverage=53.0%, noise=209

One grid step away (score change vs selected):
  eps=0.44, min_samples=2 | fewer than 2 clusters or not evaluated
  eps=0.48, min_samples=2 | score=0.1212 (-0.0153), clusters=60
  min_samples: no lower value in grid
  eps=0.46, min_samples=3 | fewer than 2 clusters or not evaluated
